<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Models

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

fatal: destination path 'DS5001-Final-Project' already exists and is not an empty directory.


In [2]:
!wget -O TFIDF_L2.csv "https://virginia.box.com/shared/static/959w70gbj2ckxew7clguvy3updzfn3t6"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"

--2026-04-14 21:48:54--  https://virginia.box.com/shared/static/959w70gbj2ckxew7clguvy3updzfn3t6
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/959w70gbj2ckxew7clguvy3updzfn3t6 [following]
--2026-04-14 21:48:54--  https://virginia.box.com/public/static/959w70gbj2ckxew7clguvy3updzfn3t6
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/959w70gbj2ckxew7clguvy3updzfn3t6 [following]
--2026-04-14 21:48:54--  https://virginia.app.box.com/public/static/959w70gbj2ckxew7clguvy3updzfn3t6
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443... connected.
HTTP r

In [3]:
import pandas as pd
TFIDF_L2 = pd.read_csv('TFIDF_L2.csv', delimiter='|', index_col=[0,1,2])
LIB = pd.read_csv('LIB.csv', delimiter='|', index_col=0)

## PCA Components

In [4]:
from sklearn.decomposition import PCA
import numpy as np

pca_engine = PCA(n_components=5)
pca_engine.fit(TFIDF_L2)

COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF_L2.columns
COMPS.index.name = 'term_str'
COMPS.head()

,PC0,PC1,PC2,PC3,PC4
term_str,,,,,
a,-0.001856,0.000813,-0.000255,-0.000915,0.000633
all,-0.004734,0.001731,0.002254,0.000077,-0.000672
and,-0.003149,0.002058,0.000808,-0.001754,0.000770
around,-0.002270,0.002348,0.000597,-0.000919,0.000180
at,-0.003821,0.002882,0.000688,-0.002092,0.000404


In [5]:
print("Top 5 positive terms for PC0:")
print(COMPS['PC0'].nlargest(5))

print("\nTop 5 negative terms for PC1:")
print(COMPS['PC1'].nsmallest(5))

Top 5 positive terms for PC0:
term_str
la           0.049102
edge         0.005064
christmas    0.004530
snippet      0.004039
chris        0.003411
Name: PC0, dtype: float64

Top 5 negative terms for PC1:
term_str
want      -0.027687
body      -0.023977
romance   -0.020529
bad       -0.014380
gaga      -0.013378
Name: PC1, dtype: float64


In [6]:
pc0_pos = COMPS['PC0'].nlargest(5)
pc0_neg = COMPS['PC0'].nsmallest(5)
pc1_pos = COMPS['PC1'].nlargest(5)
pc1_neg = COMPS['PC1'].nsmallest(5)

PCA_SUMMARY = pd.DataFrame({
    "PC0_positive_term": pc0_pos.index,
    "PC0_positive_loading": pc0_pos.values,

    "PC0_negative_term": pc0_neg.index,
    "PC0_negative_loading": pc0_neg.values,

    "PC1_positive_term": pc1_pos.index,
    "PC1_positive_loading": pc1_pos.values,

    "PC1_negative_term": pc1_neg.index,
    "PC1_negative_loading": pc1_neg.values,
})

PCA_SUMMARY.to_csv("/content/DS5001-Final-Project/PCA_SUMMARY.csv", sep="|", index=True)

## PCA DCM

In [7]:
TFIDF_L2.index = (
    TFIDF_L2.index.get_level_values('Artist') + ' — ' +
    TFIDF_L2.index.get_level_values('Title') + ' (' +
    TFIDF_L2.index.get_level_values('Album') + ')'
)
TFIDF_L2.index.name = 'track_id'

In [8]:
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_L2), index=TFIDF_L2.index)
DCM.columns = ['PC{}'.format(i) for i in DCM.columns]
DCM.head()

,PC0,PC1,PC2,PC3,PC4
track_id,,,,,
Ariana Grande — Brand New You (13 (Original Broadway Cast Recording)),0.067422,-0.015550,-0.069531,-0.006991,-0.016499
Ariana Grande — Ariana Grande Tour Guide (Ariana Grande),0.109886,-0.040870,-0.086147,0.007557,-0.014324
Ariana Grande — I’m Every Woman/Vogue (Ariana Grande),-0.002872,-0.047161,0.017458,-0.004826,-0.016378
Ariana Grande — Leave Me Lonely (Reprise) (Ariana Grande),0.081426,-0.037333,-0.057521,0.033939,-0.009213
Ariana Grande — Love Me Harder/breathin (Ariana Grande),0.077304,-0.031454,-0.060082,0.014218,-0.029139


In [9]:
DCM.to_csv('/content/DS5001-Final-Project/DCM.csv', sep='|', index=True)

## PCA Loadings

In [10]:
pca_engine = PCA(n_components=5)
pca_engine.fit(TFIDF_L2)

COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF_L2.columns
COMPS.index.name = 'term_str'
COMPS.head()

,PC0,PC1,PC2,PC3,PC4
term_str,,,,,
a,-0.001841,0.000819,-0.000266,-0.000968,0.000623
all,-0.004714,0.001794,0.002149,-0.000036,-0.000543
and,-0.003138,0.002086,0.000751,-0.001826,0.000821
around,-0.002287,0.002348,0.000545,-0.001003,0.000452
at,-0.003805,0.002929,0.000642,-0.002162,0.000454


In [11]:
COMPS.to_csv('/content/DS5001-Final-Project/COMPS.csv', sep='|', index=True)

####Package installations

In [12]:
! pip install plotly_express

In [13]:
! pip install kaleido==0.2.1

## PCA Visualization 1

In [14]:
import plotly_express as px
import plotly.io as pio
pio.renderers.default = 'colab'

def vis_pcs(DCM, a, b, label='Artist'):
    LIB_reduced = LIB.loc[DCM.index]
    fig = px.scatter(DCM, f"PC{a}", f"PC{b}",
                    color=LIB_reduced[label],
                    hover_name=LIB_reduced.index,
                    size=LIB_reduced['doc_length_words'],
                    marginal_x='box',
                    height=800,
                    title=f"PCA — PC{a} vs PC{b} colored by {label}")
    fig.update_traces(marker=dict(line=dict(width=0)))
    fig.update_layout(
        xaxis=dict(showgrid=True, gridcolor='lightgrey'),
        yaxis=dict(showgrid=True, gridcolor='lightgrey')
    )
    return fig

def vis_loadings(COMPS, a=0, b=1, hover_name='term_str'):
    fig = px.scatter(COMPS.reset_index(), f"PC{a}", f"PC{b}",
                      text='term_str',
                      marginal_x='box',
                      height=800,
                      title=f"PCA Loadings — PC{a} vs PC{b}")
    fig.update_traces(marker=dict(line=dict(width=0)))
    fig.update_layout(
        xaxis=dict(showgrid=True, gridcolor='lightgrey'),
        yaxis=dict(showgrid=True, gridcolor='lightgrey')
    )
    return fig

In [15]:
import kaleido
fig1 = vis_pcs(DCM, 0, 1)
pio.write_image(fig1, "/content/DS5001-Final-Project/images/PCA_components_PC0_PC1.png", format="png", scale=2, engine="kaleido")
fig1

In [16]:
fig2 = vis_loadings(COMPS, 0, 1)
pio.write_image(fig2, "/content/DS5001-Final-Project/images/PCA_loadings_PC0_PC1.png", format="png", scale=2, engine="kaleido")
fig2

In [17]:
print("Positive pole (right side):")
print(COMPS['PC0'].nlargest(10))

print("\nNegative pole (left side):")
print(COMPS['PC0'].nsmallest(10))

Positive pole (right side):
term_str
la           0.049152
edge         0.005278
christmas    0.004449
snippet      0.003953
chris        0.003458
fa           0.003291
martin       0.003217
lyrics       0.003215
gaga         0.002922
light        0.002807
Name: PC0, dtype: float64

Negative pole (left side):
term_str
love    -0.008751
never   -0.008390
let     -0.008274
need    -0.008100
dont    -0.008027
baby    -0.007764
if      -0.007516
ill     -0.007404
you     -0.007365
girl    -0.007116
Name: PC0, dtype: float64


##PCA Visualization 2

In [18]:
fig3 = vis_pcs(DCM, 1, 2)
pio.write_image(fig3, "/content/DS5001-Final-Project/images/PCA_components_PC1_PC2.png", format="png", scale=2, engine="kaleido")
fig3

In [19]:
fig4 = vis_loadings(COMPS, 1, 2)
pio.write_image(fig4, "/content/DS5001-Final-Project/images/PCA_loadings_PC1_PC2.png", format="png", scale=2, engine="kaleido")
fig4

In [20]:
print("Positive pole (right side):")
print(COMPS['PC1'].nlargest(10))

print("\nNegative pole (left side):")
print(COMPS['PC1'].nsmallest(10))

Positive pole (right side):
term_str
la       0.029055
was      0.006617
never    0.006611
will     0.005564
ill      0.004818
were     0.004668
back     0.004052
are      0.003932
away     0.003891
youll    0.003872
Name: PC1, dtype: float64

Negative pole (left side):
term_str
want      -0.027686
body      -0.023523
romance   -0.020552
bad       -0.014289
gaga      -0.013270
lady      -0.009026
eh        -0.008725
what      -0.007482
hey       -0.005931
revenge   -0.005791
Name: PC1, dtype: float64


##LDA Topic